# 07 · Cached generation: speed, and the coding baseline for RL

RL generates far more text than any evaluation so far, so generation has to be fast first. This notebook

1. measures generation with and without the cache (`GLMLadderModel.new_cache`) at several batch sizes, and
2. runs the code evaluation (40 tasks, chat prompts, 1 greedy + 10 sampled answers) on the two RL starting models, `mix-s50m-lr1e-04` and `mix-m100m-lr1e-04`. Their pass rates are the numbers RL has to improve.

**Runtime:** *Select Kernel → Colab → New Colab Server → **A100***. About 20 minutes.

## 1. GPU

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 2. Free the GPU

Must run before anything touches CUDA in this kernel.

In [ ]:
# In VS Code + Colab, "Restart Kernel" can leave the old kernel (and its compile workers)
# running and holding GPU memory, which later causes CUDA out-of-memory errors.
import os
import signal

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation

me = os.getpid()
ps = subprocess.run(["ps", "-eo", "pid,args"], capture_output=True, text=True).stdout.splitlines()[1:]
stale = []
for line in ps:
    pid_text, _, args = line.strip().partition(" ")
    pid = int(pid_text)
    if pid == me:
        continue
    if "colab_kernel_launcher" in args or ("compile_worker" in args and f"--parent={me}" not in args):
        stale.append((pid, args[:80]))
for pid, args in stale:
    os.kill(pid, signal.SIGKILL)
    print("killed stale process", pid, args)
print("stale processes:", len(stale))
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 3. Get the code

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Tbt-code/GLM-Ladder-from-Scratch.git"
BRANCH = "main"
REPO_DIR = Path("/content/glm-ladder-from-scratch")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--quiet", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPO_DIR / "requirements.txt")], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", commit)

## 4. Run all tests

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(result.stdout[-2500:], result.stderr[-1000:])
assert result.returncode == 0, "tests failed - fix before training"

## 5. Log in to Hugging Face

The token is typed into a hidden prompt and lives only in this session's memory.

In [ ]:
from getpass import getpass

from huggingface_hub import HfApi

DATA_REPO = "Tbt-AI/glm-ladder-data"
RUNS_REPO = "Tbt-AI/glm-ladder-runs"

token = getpass("Hugging Face write token: ")
api = HfApi(token=token)
print("logged in as:", api.whoami()["name"])
for repo_id, repo_type in ((DATA_REPO, "dataset"), (RUNS_REPO, "model")):
    assert api.repo_info(repo_id, repo_type=repo_type).private, f"{repo_id} must be private"
    print("private:", repo_id)

## 6. Models and tasks

In [ ]:
import json
import time

import torch

sys.path.insert(0, str(REPO_DIR / "scripts"))
from glm_ladder.chat import prompt_ids
from glm_ladder.code_eval import load_tasks, sample_many
from glm_ladder.tokenizer import load_tokenizer
from run_code_eval import evaluate, receipt   # scripts/run_code_eval.py
from sample import load_run                    # scripts/sample.py: downloads, checks SHA-256

RL_START = {"s50m": "mix-s50m-lr1e-04", "m100m": "mix-m100m-lr1e-04"}
device = torch.device("cuda")
torch.set_float32_matmul_precision("high")
GPU = torch.cuda.get_device_name()
tokenizer = load_tokenizer(REPO_DIR / "tokenizer" / "glm-ladder-16k.json")
tasks = load_tasks(REPO_DIR / "evals" / "code_tasks.jsonl")
models = {}
for size, run in RL_START.items():
    model, state = load_run(api, RUNS_REPO, run, None)
    models[size] = (model.to(device).eval(), state)
    print(f"{size}: {run} @ {state['checkpoint']}")

## 7. Generation speed

Same prompt (a coding request, ~60 tokens), 160 new tokens per sequence, no early stop, temperature 0.8. *Uncached* re-runs the whole sequence for every new token, as every earlier evaluation did; *cached* runs the prompt once and then one token per step. The uncached run is skipped at the largest batch, where it would take minutes.

In [ ]:
class NoCache(torch.nn.Module):
    """The same model without new_cache(), so sample_many takes the uncached path."""

    def __init__(self, model):
        super().__init__()
        self.model, self.config = model, model.config

    def forward(self, ids):
        return self.model(ids)


prompt = prompt_ids(tokenizer, [{"role": "user", "content": tasks[0].chat_prompt()}])
NEW = 160
speed = []
for size, (model, _) in models.items():
    for n in (1, 8, 32, 128):
        row = {"size": size, "n": n}
        for name, m in (("cached", model), ("uncached", NoCache(model))):
            if name == "uncached" and n > 32:
                continue
            settings = dict(n=n, max_new_tokens=NEW, temperature=0.8, top_k=50, seed=0, stop=lambda _: False)
            sample_many(m, prompt, **{**settings, "max_new_tokens": 4})        # warm-up
            torch.cuda.synchronize()
            started = time.perf_counter()
            sample_many(m, prompt, **settings)
            torch.cuda.synchronize()
            row[name] = round(n * NEW / (time.perf_counter() - started))
        speed.append(row)
        print(f"{size:6s} n={n:4d}  cached {row['cached']:>8,} tok/s" +
              (f"  uncached {row['uncached']:>8,} tok/s  speed-up {row['cached'] / row['uncached']:.1f}x" if "uncached" in row else ""))

## 8. Coding baseline of the RL starting models

The same evaluation as notebook 04 (chat mode), now with the cache. For comparison, the SFT models scored greedy pass@1 0.375 / 0.475 and sampled pass@10 0.50 / 0.65 (`s-50m` / `m-100m`).

In [ ]:
SETTINGS = dict(n=10, temperature=0.8, top_k=50, max_tokens=160, seed=0, loss_only=False)
OUT = Path("/content/evals")
OUT.mkdir(exist_ok=True)
summaries = {}
for size, (model, state) in models.items():
    run = RL_START[size]
    started = time.perf_counter()
    results, summary = evaluate(model, tokenizer, tasks, mode="chat", log=lambda _: None, **SETTINGS)
    data = receipt(run, state["checkpoint"], "chat", results, summary, device=GPU, **SETTINGS)
    data["commit"] = commit
    (OUT / f"code-{run}-chat.json").write_text(json.dumps(data, indent=2) + "\n", encoding="utf-8")
    summaries[run] = summary
    print(f"{run:20s} {time.perf_counter() - started:5.0f} s  " + "  ".join(f"{k} {v:.3f}" for k, v in summary.items()))

## 9. Upload

In [ ]:
(OUT / "generation-speed.json").write_text(json.dumps({"schema_version": "1.0", "kind": "generation_speed", "gpu": GPU,
                                                     "commit": commit, "new_tokens": NEW, "rows": speed}, indent=2) + "\n")
api.upload_folder(folder_path=str(OUT), path_in_repo="evals", repo_id=RUNS_REPO, repo_type="model",
                  commit_message=f"cached generation speed and RL-start code evals, commit {commit}")
print("uploaded")

## 10. Done

Close the runtime. Locally:

```bash
python scripts/fetch_evals.py
```